# Modul 9 · Bäume, Ensembles und Feature Engineering
**Data Science 1** · linkit courses

Warum gewinnen Baum-Ensembles bei Tabellendaten so oft? Du teilst Chargen der Härterei von Hand, liest einen Klassifikationsbaum, baust Bagging und Boosting selbst nach, stoppst ein Boosting zeitlich, baust rollierende Merkmale für die Sensordaten und eine Leckage dazu, suchst Hyperparameter fair, misst Wichtigkeiten und prüfst, wo Bäume an ihre Grenzen kommen.

**Lernziele**
1. Einen Entscheidungsbaum aufbauen, lesen und seine Überanpassung erklären
2. Bagging und Random Forest als Mittel gegen Streuung verstehen
3. Gradient Boosting mit Lernrate und frühem Stoppen anwenden
4. Merkmale für Sensordaten zeitlich korrekt und ohne Leckage bauen
5. Hyperparameter mit zeitlicher Kreuzvalidierung fair auswählen
6. Wichtigkeit von Merkmalen messen und vorsichtig deuten
7. Begründen, wann Baum-Ensembles gewinnen und wann nicht

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Der erste Import von scikit-learn lädt im Browser SciPy und scikit-learn, das dauert einmalig etwa 10 Sekunden.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt.
- Mindestziel sind TODO 1 bis 4, TODO 6 bis 11 sowie TODO 13. TODO 5, TODO 12 und die Bonusaufgaben vertiefen.
- TODO-Zellen mit Zufall legen den Seed fest, etwa mit `np.random.default_rng(7)` oder `random_state=3`. Die Selbsttests lassen so viel Spielraum, dass jeder richtige Weg besteht.
- Gib Ergebnisse mit `float(...)`, `round(...)` oder der Hilfsfunktion `zeige` aus, dann steht im Browser und in Colab dasselbe da. Jede Grafikzelle endet mit `plt.show()`.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["haerterei.csv", "sensordaten.csv", "maschinen.csv", "produktion.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Ein Baum, der Fragen stellt
*Lerneinheit 1 · Skript Abschnitt 1*

Ein **Regressionsbaum** teilt die Daten mit Fragen wie „Transferzeit ≤ 19,15 s?“. Für jede mögliche Schwelle $s$ eines Merkmals misst er die Summe der quadrierten Abweichungen in beiden Teilen,

$$Q(s) = \sum_{i \in L} (y_i - \bar y_L)^2 + \sum_{i \in R} (y_i - \bar y_R)^2,$$

und nimmt die kleinste. Als Schwellen kommen die Mitten zwischen benachbarten verschiedenen Werten in Frage. Ein **Klassifikationsbaum** nimmt statt $Q$ die Gini-Unreinheit $2p(1 - p)$ und sagt im Blatt den Anteil voraus. Ohne Grenze wächst ein Baum, bis er die Trainingsdaten auswendig kennt.

Die nächste Zelle lädt die Härterei wie im Skript, mit der Öltemperatur auf ganze Grad gerundet: `haerterei` hat eine Zeile je Ofencharge, `OFEN` nennt die sechs Einstellungen, `h1` ist das erste Jahr bis zum 30. September 2025 zum Trainieren, `h2` das zweite zum Prüfen. Zielgrößen sind die Härte `haerte_hrc` und `in_toleranz` (52 bis 58 HRC). Die Hilfsfunktion `zeige` druckt Zahlen mit 6 gültigen Stellen, Tabellen mit ihrer Größe und lange Arrays mit Länge und Summe.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, export_text
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score

SCHICHT_TYP = pd.CategoricalDtype(["Früh", "Spät", "Nacht"], ordered=True)
haerterei = pd.read_csv("daten/haerterei.csv", dtype={"schicht": SCHICHT_TYP})
haerterei["oeltemp_c"] = haerterei["oeltemp_c"].round()
OFEN = ["ofentemp_c", "haltezeit_min", "chargengewicht_kg", "kohlenstoff_pct", "oeltemp_c", "transfer_s"]
h1 = haerterei[haerterei["datum"] < "2025-10-01"]
h2 = haerterei[haerterei["datum"] >= "2025-10-01"]
print(haerterei.shape, len(h1), len(h2), round(haerterei["haerte_hrc"].mean(), 2), round(haerterei["in_toleranz"].mean(), 4))


def zeige(*werte):
    """Druckt Zahlen mit 6 gültigen Stellen, damit im Browser und in Colab dieselben Ziffern stehen.
    Tabellen erscheinen mit ihrer Größe, lange Arrays mit Länge und Summe. None bleibt None."""
    def text(w):
        if w is None or isinstance(w, (str, bool)):
            return str(w)
        if isinstance(w, dict):
            return "{" + ", ".join(f"{k}: {text(v)}" for k, v in w.items()) + "}"
        if isinstance(w, pd.DataFrame):
            return f"Tabelle {w.shape[0]} × {w.shape[1]}"
        if np.ndim(w) == 0:
            return f"{float(w):.6g}"
        w = np.ravel(w)
        if len(w) > 12:
            return f"[{len(w)} Werte, Summe {float(np.sum(w)):.6g}]"
        return "[" + " ".join(f"{float(v):.6g}" for v in w) + "]"
    print(*[text(w) for w in werte])

Der Baum der Tiefe 2 aus dem Skript, gelesen mit `export_text`:

In [ ]:
baum2 = DecisionTreeRegressor(max_depth=2, random_state=0).fit(h1[OFEN], h1["haerte_hrc"])
print(export_text(baum2, feature_names=OFEN))

### TODO 1 · Die beste Teilung von Hand
Das Skript teilt acht Chargen nach der Transferzeit. Hier teilst du die 1.410 Chargen des ersten Jahres nach der Haltezeit, für die Härte.
1. `kandidaten_halte`: alle Mitten zwischen benachbarten verschiedenen Werten von `haltezeit_min` in `h1`, aufsteigend, als Array. Tipp: `np.unique` sortiert und entfernt Doppelte.
2. `qs_halte`: für jede Schwelle aus `kandidaten_halte` die Summe $Q(s)$ der quadrierten Abweichungen der Härte in beiden Teilen, als Array. `schwelle_hand`: die Schwelle mit dem kleinsten $Q$. `mittel_links` und `mittel_rechts`: die mittlere Härte der Chargen mit Haltezeit bis zu dieser Schwelle und darüber.
3. `stumpf_halte`: `DecisionTreeRegressor(max_depth=1, random_state=0)`, angepasst mit `h1[["haltezeit_min"]]` und der Härte. Seine Schwelle steht in `stumpf_halte.tree_.threshold[0]`.

In [ ]:
x_halte = h1["haltezeit_min"].to_numpy()
y_haerte = h1["haerte_hrc"].to_numpy()

# TODO 1a
kandidaten_halte = None

# TODO 1b
qs_halte = None
schwelle_hand = None
mittel_links = None
mittel_rechts = None

# TODO 1c
stumpf_halte = None

zeige(kandidaten_halte, schwelle_hand, mittel_links, mittel_rechts, None if qs_halte is None else np.min(qs_halte))

In [ ]:
check("TODO 1a Kandidaten", lambda: len(kandidaten_halte) == 82 and kandidaten_halte[0] == 36.0 and kandidaten_halte[-1] == 125.0 and np.all(np.diff(kandidaten_halte) > 0))
check("TODO 1b beste Teilung von Hand", lambda: len(qs_halte) == 82 and math.isclose(float(np.min(qs_halte)), 10850.87137154, abs_tol=1e-6)
      and schwelle_hand == 73.5 and math.isclose(mittel_links, 54.52484076, abs_tol=1e-7) and math.isclose(mittel_rechts, 56.13337596, abs_tol=1e-7))
check("TODO 1c Stumpf wie von Hand", lambda: math.isclose(stumpf_halte.tree_.threshold[0], schwelle_hand, abs_tol=1e-4)
      and np.allclose(stumpf_halte.predict(pd.DataFrame({"haltezeit_min": [60, 90]})), [mittel_links, mittel_rechts]))

Von 82 möglichen Schwellen ist 73,5 min die beste: Chargen mit kürzerer Haltezeit haben im Mittel 54,52 HRC, mit längerer 56,13. Die Summe der quadrierten Abweichungen sinkt von 11.752,0 auf 10.850,9, nur um 7,7 %. Die Transferzeit verringert sie in der ersten Teilung des Skripts weit stärker, deshalb fragt der Baum dort zuerst nach ihr. Der Stumpf aus scikit-learn findet dieselbe Schwelle.

### TODO 2 · Ein Klassifikationsbaum für die Toleranz
1. `baum_tol`: `DecisionTreeClassifier(max_depth=3, random_state=0)` für `in_toleranz` aus den sechs Merkmalen `OFEN` im ersten Jahr. Lies ihn mit `print(export_text(baum_tol, feature_names=OFEN, show_weights=True))`. Unter `weights` steht je Blatt die Zahl der Chargen außerhalb und innerhalb der Toleranz.
2. `schwelle_wurzel`: die Schwelle der ersten Frage. Im Baum gibt es ein Blatt für Chargen mit Kohlenstoff über 0,4675 % und Transferzeit bis 18,35 s, `export_text` rundet die Schwelle des Kohlenstoffs auf 0,47. `n_hart`: die Zahl der Chargen des ersten Jahres in diesem Blatt, `quote_hart`: der Anteil davon in Toleranz.
3. `auc_baum_tol`: die AUC des Baums im zweiten Jahr. `regel`: ein Satz in Werkssprache über dieses Blatt, als Text, in dem das Wort „Kohlenstoff“ vorkommt.

In [ ]:
# TODO 2a
baum_tol = None

# TODO 2b
schwelle_wurzel = None
n_hart = None
quote_hart = None

# TODO 2c
auc_baum_tol = None
regel = None

zeige(schwelle_wurzel, n_hart, quote_hart, auc_baum_tol)
print(regel)

In [ ]:
check("TODO 2a Baum der Tiefe 3", lambda: type(baum_tol).__name__ == "DecisionTreeClassifier" and baum_tol.get_depth() == 3 and baum_tol.get_n_leaves() == 7)
check("TODO 2b Wurzel und hartes Blatt", lambda: math.isclose(schwelle_wurzel, 22.85, abs_tol=1e-4) and n_hart == 180 and math.isclose(quote_hart, 0.4, abs_tol=1e-9))
check("TODO 2c AUC und Regel", lambda: math.isclose(auc_baum_tol, 0.70508046, abs_tol=1e-6) and isinstance(regel, str) and "Kohlenstoff" in regel and len(regel.split()) >= 8)

Die Toleranz wird auf zwei Arten verfehlt. Über 22,85 s Transferzeit sind die Chargen zu weich, bei Ofentemperatur bis 869,5 °C liegt dort keine einzige von 45 in der Toleranz. Bei viel Kohlenstoff und schnellem Transfer sind sie zu hart, nur 72 von 180 liegen darin. Mit sieben Blättern erreicht der Baum im zweiten Jahr eine AUC von 0,705, etwas weniger als die logistische Regression mit 0,732. Er gibt nur sechs verschiedene Wahrscheinlichkeiten aus, zwei Blätter sagen 0 voraus.

### TODO 3 · Güte gegen Tiefe
1. `auc_tiefe`: eine Tabelle mit dem Index 1 bis 10 für die Tiefe und den Spalten `jahr1` und `jahr2`: die AUC eines `DecisionTreeClassifier(max_depth=..., random_state=0)` für `in_toleranz` im ersten und im zweiten Jahr, angepasst mit dem ersten.
2. `luecke10`: um wie viel die AUC bei Tiefe 10 im ersten Jahr höher liegt als im zweiten.

In [ ]:
# TODO 3a
auc_tiefe = None

# TODO 3b
luecke10 = None

zeige(auc_tiefe, luecke10)

In [ ]:
check("TODO 3a AUC je Tiefe", lambda: list(auc_tiefe.index) == list(range(1, 11))
      and np.allclose(auc_tiefe["jahr1"], [0.59862409, 0.70983633, 0.7618436, 0.79911385, 0.84579847, 0.87823244, 0.90719144, 0.92759013, 0.9478506, 0.96568185], atol=1e-6)
      and np.allclose(auc_tiefe["jahr2"], [0.59081887, 0.67226267, 0.70508046, 0.72862825, 0.78372542, 0.78245828, 0.75983711, 0.74758218, 0.74543652, 0.72120769], atol=1e-6))
check("TODO 3b Lücke bei Tiefe 10", lambda: math.isclose(luecke10, 0.24447416, abs_tol=1e-6))

In [ ]:
if auc_tiefe is None:
    print("⏳ Erst TODO 3 lösen.")
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(auc_tiefe.index, auc_tiefe["jahr1"], "o-", label="erstes Jahr (Training)")
    ax.plot(auc_tiefe.index, auc_tiefe["jahr2"], "o-", label="zweites Jahr (Prüfung)")
    ax.set_xlabel("Tiefe des Baums")
    ax.set_ylabel("AUC für in_toleranz")
    ax.legend()
    plt.show()

Im ersten Jahr steigt die AUC mit jeder Ebene bis 0,966. Im zweiten erreicht sie bei Tiefe 5 ihr Maximum von 0,784 und fällt dann auf 0,721, die Lücke bei Tiefe 10 beträgt 0,244. Die beste Tiefe nach dem zweiten Jahr zu wählen, wäre aber schon eine Anpassung an die Prüfdaten. Dafür gibt es in Abschnitt 5 die Kreuzvalidierung.

## 2 · Viele Bäume: Bagging und Random Forest
*Lerneinheit 2 · Skript Abschnitt 2*

**Bagging** zieht $B$ Bootstrap-Stichproben mit Zurücklegen, passt auf jeder einen tiefen Baum an und mittelt die Vorhersagen. Bei Klassifikation werden die Wahrscheinlichkeiten gemittelt. Der **Random Forest** lässt jede Teilung nur unter einer zufälligen Auswahl von `max_features` Merkmalen suchen. Jeder Baum hat etwa 36,8 % der Zeilen nie gesehen, daraus entsteht die **Out-of-Bag-Schätzung**.

Die beiden Wälder aus dem Skript: `rf` für die Härte und `rf_tol` für die Toleranz, beide mit dem ersten Jahr angepasst.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier

rf = RandomForestRegressor(n_estimators=200, max_features=3, min_samples_leaf=5, oob_score=True, random_state=1).fit(h1[OFEN], h1["haerte_hrc"])
rf_tol = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, oob_score=True, random_state=1).fit(h1[OFEN], h1["in_toleranz"])
print(round(rf.oob_score_, 3), round(r2_score(h2["haerte_hrc"], rf.predict(h2[OFEN])), 3))
print(round(roc_auc_score(h1["in_toleranz"], rf_tol.oob_decision_function_[:, 1]), 4), round(roc_auc_score(h2["in_toleranz"], rf_tol.predict_proba(h2[OFEN])[:, 1]), 4))

### TODO 4 · Bagging von Hand
Das Skript mittelt Bäume für die Härte. Hier mittelst du Klassifikationsbäume für die Toleranz. `zufall4` ist ein Zufallsgenerator mit festem Seed.
1. `p_bag_hand`: das Mittel der Wahrscheinlichkeiten für `in_toleranz` von 25 tiefen Bäumen `DecisionTreeClassifier(random_state=0)` für die Chargen des zweiten Jahres. Jeder Baum lernt auf einer eigenen Bootstrap-Stichprobe der Zeilen von `h1`: `zeilen_ = zufall4.integers(0, len(h1), len(h1))`, dann `h1.iloc[zeilen_]`. `auc_bag_hand`: die AUC von `p_bag_hand` im zweiten Jahr.
2. `auc_ein_baum`: die AUC eines einzelnen tiefen Baums, angepasst mit dem ganzen ersten Jahr.
3. `auc_bag_sk`: die AUC von `BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=25, random_state=7)`.

In [ ]:
zufall4 = np.random.default_rng(7)

# TODO 4a
p_bag_hand = None
auc_bag_hand = None

# TODO 4b
auc_ein_baum = None

# TODO 4c
auc_bag_sk = None

zeige(auc_bag_hand, auc_ein_baum, auc_bag_sk)

In [ ]:
check("TODO 4a Bagging von Hand", lambda: len(p_bag_hand) == 1406 and 0.845 <= auc_bag_hand <= 0.885 and len(np.unique(p_bag_hand)) > 25)
check("TODO 4b ein einzelner Baum", lambda: math.isclose(auc_ein_baum, 0.71553025, abs_tol=1e-6))
check("TODO 4c Bagging mit scikit-learn", lambda: 0.845 <= auc_bag_sk <= 0.885 and auc_bag_hand - auc_ein_baum > 0.12)

Ein einzelner tiefer Baum kommt im zweiten Jahr auf eine AUC von 0,716, 25 gemittelte auf etwa 0,86, von Hand wie mit `BaggingClassifier`. Ein tiefer Baum gibt fast nur 0 und 1 aus. Das Mittel aus 25 Bäumen ordnet die Chargen viel feiner.

Für TODO 5 kommen die Sensordaten aus Modul 8 dazu, mit der Hallentemperatur aus `produktion.csv` und den Pipelines für Bäume wie im Skript. `jahr1` und `jahr2` teilen nach der Zeit, `ROH` nennt die drei Messgrößen. Abschnitt 4 ergänzt die neuen Merkmale.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import average_precision_score

sensoren = pd.read_csv("daten/sensordaten.csv", dtype={"schicht": SCHICHT_TYP})
maschinen = pd.read_csv("daten/maschinen.csv")
produktion = pd.read_csv("daten/produktion.csv", dtype={"schicht": SCHICHT_TYP})
daten = sensoren.merge(maschinen[["maschine", "linie", "typ"]], on="maschine", validate="many_to_one")
daten = daten.merge(produktion[["datum", "schicht", "linie", "hallentemp_c"]], on=["datum", "schicht", "linie"], how="left", validate="many_to_one")
ROH = ["vibration_mm_s", "lagertemp_c", "strom_pct"]
jahr1 = daten[daten["datum"] < "2025-10-01"]
jahr2 = daten[daten["datum"] >= "2025-10-01"]
y1, y2 = jahr1["ausfall_naechste"].to_numpy(), jahr2["ausfall_naechste"].to_numpy()


def baumpipeline(modell, spalten):
    vv = make_column_transformer((SimpleImputer(strategy="median"), spalten), (OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["typ"]))
    return make_pipeline(vv, modell)


print(daten.shape, len(jahr1), len(jahr2))

### TODO 5 · Out-of-Bag gegen das zweite Jahr
Bei der Härterei trifft die Out-of-Bag-Schätzung das zweite Jahr. Bei Zeitdaten kennen die Bäume, die eine Schicht nicht gesehen haben, meist ihre Nachbarschichten. Was passiert, wenn das Datum selbst ein Merkmal ist?
1. `mit_tag`: eine Kopie von `daten` mit der Spalte `tag`, der Zahl der Tage seit dem 1. Oktober 2024 als ganze Zahl. Tipp: `(pd.to_datetime(...) - pd.Timestamp("2024-10-01")).dt.days`.
2. Für die Merkmale `ROH` und für `ROH + ["tag"]` je ein Wald `baumpipeline(RandomForestClassifier(n_estimators=50, min_samples_leaf=5, oob_score=True, random_state=3), spalten)`, angepasst mit den Zeilen des ersten Jahres (`mit_tag.loc[jahr1.index, spalten + ["typ"]]`). `oob_auc` und `auc_j2`: Wörterbücher mit den Schlüsseln `"ohne"` und `"mit"`, darin die AUC aus `oob_decision_function_[:, 1]` im ersten Jahr und die AUC im zweiten Jahr. Den Wald in der Pipeline erreichst du mit `[-1]`.

In [ ]:
# TODO 5a
mit_tag = None

# TODO 5b
oob_auc = None
auc_j2 = None

zeige(oob_auc, auc_j2)

In [ ]:
check("TODO 5a Tage seit Beginn", lambda: len(mit_tag) == 28920 and int(mit_tag["tag"].min()) == 0 and int(mit_tag["tag"].max()) == 729 and int(mit_tag["tag"].sum()) == 10491960)
check("TODO 5b Out-of-Bag-AUC", lambda: 0.705 <= oob_auc["ohne"] <= 0.745 and 0.715 <= oob_auc["mit"] <= 0.763 and 0.716 <= auc_j2["ohne"] <= 0.742 and 0.705 <= auc_j2["mit"] <= 0.745)
check("TODO 5b Datum täuscht die OOB-Schätzung", lambda: (oob_auc["mit"] - auc_j2["mit"]) - (oob_auc["ohne"] - auc_j2["ohne"]) > 0)

Mit dem Datum als Merkmal steigt die Out-of-Bag-AUC mit Seed 3 von 0,729 auf 0,733, im zweiten Jahr fällt sie von 0,727 auf 0,719. Der Wald merkt sich, in welchen Wochen eine Maschine oft ausfiel. Out-of-Bag prüft er das an Schichten aus genau diesen Wochen, im zweiten Jahr gibt es keine solchen Tage. Bei Zeitdaten prüft nur die Zukunft ehrlich.

## 3 · Boosting: aus Fehlern lernen
*Lerneinheit 3 · Skript Abschnitt 3*

Gradient Boosting beginnt mit dem Mittelwert $f_0 = \bar y$ und passt in jeder Runde einen kleinen Baum $h_m$ an die Residuen $y_i - f_{m-1}(x_i)$ an: $f_m = f_{m-1} + \nu\, h_m$ mit der Lernrate $\nu$. Wann aufgehört wird, entscheidet der Fehler auf Validierungsdaten. Bei `HistGradientBoosting*` setzt du `early_stopping` ausdrücklich und validierst bei Zeitdaten selbst mit einem späteren Zeitraum.

### TODO 6 · Boosting von Hand mit Stümpfen
Das Skript rechnet drei Runden an acht Chargen. Hier rechnest du zwei Runden an allen Chargen des ersten Jahres, mit allen sechs Merkmalen und der Lernrate 0,1. `y_haerte` aus TODO 1 legt die nächste Zelle noch einmal an.
1. `f0`: ein Array mit der mittleren Härte für jede Charge. `stumpf_1`: `DecisionTreeRegressor(max_depth=1, random_state=0)`, angepasst mit `h1[OFEN]` und den Residuen `y_h - f0`. `f1`: `f0` plus 0,1 mal die Vorhersage von `stumpf_1`.
2. `stumpf_2` und `f2` genauso für die zweite Runde.
3. `mse_runden`: ein Array mit dem mittleren quadrierten Fehler im ersten Jahr nach Runde 0, 1 und 2. `abweichung`: die größte absolute Differenz zwischen `f2` und den Vorhersagen von `GradientBoostingRegressor(n_estimators=2, learning_rate=0.1, max_depth=1, random_state=0)` für das erste Jahr.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

y_h = h1["haerte_hrc"].to_numpy()

# TODO 6a
f0 = None
stumpf_1 = None
f1 = None

# TODO 6b
stumpf_2 = None
f2 = None

# TODO 6c
mse_runden = None
abweichung = None

zeige(mse_runden, abweichung)

In [ ]:
check("TODO 6a erste Runde", lambda: OFEN[stumpf_1.tree_.feature[0]] == "transfer_s" and math.isclose(stumpf_1.tree_.threshold[0], 19.15, abs_tol=1e-4)
      and np.allclose(np.unique(f1), [54.99001927, 55.50146602], atol=1e-6))
check("TODO 6b zweite Runde", lambda: OFEN[stumpf_2.tree_.feature[0]] == "transfer_s" and math.isclose(stumpf_2.tree_.threshold[0], 18.85, abs_tol=1e-4))
check("TODO 6c Fehler je Runde wie scikit-learn", lambda: np.allclose(mse_runden, [8.33478361, 7.64921864, 7.0927244], atol=1e-6) and abweichung < 1e-9)

Beide Stümpfe fragen nach der Transferzeit, der erste bei 19,15 s, der zweite bei 18,85 s. Der mittlere quadrierte Fehler sinkt von 8,335 auf 7,649 und 7,093. Mit der Lernrate 0,1 korrigiert jede Runde nur ein Zehntel der Residuen, deshalb braucht Boosting viele Runden. `GradientBoostingRegressor` rechnet genau dasselbe.

### TODO 7 · Boosting für die Toleranz, zeitlich gestoppt
Das Skript stoppt ein Boosting für die Härte. Hier stoppst du eines für die Toleranz, validiert mit den letzten drei Monaten des ersten Jahres. Als Fehler dient der Log-Loss aus Modul 8.
1. `hgb_lang`: `HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300, early_stopping=False, random_state=0)`, angepasst mit `lern7`. `logloss_val`: ein Array mit dem Log-Loss auf `val7` nach jeder der 300 Runden. Tipp: `hgb_lang.staged_predict_proba(val7[OFEN])` liefert die Wahrscheinlichkeiten Runde für Runde.
2. `n_runden`: die Runde mit dem kleinsten Log-Loss, gezählt ab 1.
3. `hgb_tol`: dasselbe Modell mit `max_iter=n_runden`, angepasst mit dem ganzen ersten Jahr. `auc_hgb_tol`: seine AUC im zweiten Jahr.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss

lern7 = h1[h1["datum"] < "2025-07-01"]
val7 = h1[h1["datum"] >= "2025-07-01"]

# TODO 7a
hgb_lang = None
logloss_val = None

# TODO 7b
n_runden = None

# TODO 7c
hgb_tol = None
auc_hgb_tol = None

zeige(n_runden, None if logloss_val is None else np.min(logloss_val), auc_hgb_tol)

In [ ]:
check("TODO 7a Log-Loss je Runde", lambda: len(logloss_val) == 300 and math.isclose(float(np.min(logloss_val)), 0.41068004, abs_tol=1e-6) and logloss_val[-1] > np.min(logloss_val) + 0.05)
check("TODO 7b Runde mit dem kleinsten Fehler", lambda: n_runden == 23)
check("TODO 7c gestopptes Modell", lambda: hgb_tol.n_iter_ == 23 and math.isclose(auc_hgb_tol, 0.87481518, abs_tol=1e-6))

Der Log-Loss auf den Monaten Juli bis September ist nach 23 Runden am kleinsten und steigt danach wieder, nach 300 Runden liegt er deutlich höher. Auf dem ganzen ersten Jahr mit 23 Runden angepasst, erreicht das Boosting im zweiten Jahr eine AUC von 0,875, der einzelne Baum aus TODO 2 nur 0,705. Mit `early_stopping=True` hätte scikit-learn 10 % zufällige Chargen von `lern7` zur Validierung genommen und mit `random_state=0` nach 46 Runden gestoppt, mit `random_state=1` schon nach 38. Mit der Voreinstellung `"auto"` stoppt es bei weniger als 10.000 Zeilen gar nicht.

## 4 · Merkmale bauen: Feature Engineering
*Lerneinheit 4 · Skript Abschnitt 4*

Für Sensordaten sind Merkmale relativ zum **Normalwert der eigenen Maschine** oft mehr wert als ein flexibleres Modell. Ein rollierender Normalwert nutzt nur die eigene Vergangenheit: `groupby("maschine")`, dann `shift(1)`, dann `rolling(...)`. Ohne `shift` steckt die aktuelle Schicht im eigenen Vergleichswert, mit `center=True` die Zukunft, ohne `groupby` die Nachbarmaschinen. Der **Abschneidetest** findet Merkmale, die in die Zukunft schauen: Für Zeilen vor einem Stichtag muss ein Merkmal gleich bleiben, wenn es nur mit den Daten bis dahin berechnet wird.

Die nächste Zelle baut die neuen Merkmale wie im Skript, mit einem Fenster von 30 Schichten, und die drei Modelle aus Abschnitt 4.4. Das Boosting ist dort `GradientBoostingClassifier` mit Bäumen der Tiefe 2.

In [ ]:
gruppe = daten.groupby("maschine")
for spalte, kurz in [("vibration_mm_s", "vib"), ("lagertemp_c", "temp"), ("strom_pct", "strom")]:
    normal = gruppe[spalte].transform(lambda x: x.shift(1).rolling(30, min_periods=10).median())
    daten[kurz + "_rel"] = (daten[spalte] / normal - 1).round(4)
daten["vib_diff"] = gruppe["vibration_mm_s"].diff().round(2)
daten["temp_halle"] = (daten["lagertemp_c"] - daten["hallentemp_c"]).round(1)
NEU = ["vib_rel", "temp_rel", "strom_rel", "vib_diff", "temp_halle"]
jahr1 = daten[daten["datum"] < "2025-10-01"]
jahr2 = daten[daten["datum"] >= "2025-10-01"]
y1, y2 = jahr1["ausfall_naechste"].to_numpy(), jahr2["ausfall_naechste"].to_numpy()
l2_vor = ((jahr2["linie"] == 2) & (jahr2["datum"] < "2026-03-02")).to_numpy()
l2_nach = ((jahr2["linie"] == 2) & (jahr2["datum"] >= "2026-03-02")).to_numpy()
modelle = {"logistisch": lambda s: make_pipeline(make_column_transformer((make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), s),
                                                                          (OneHotEncoder(handle_unknown="ignore"), ["typ"])), LogisticRegression(solver="newton-cholesky")),
           "Random Forest": lambda s: baumpipeline(RandomForestClassifier(n_estimators=100, min_samples_leaf=50, oob_score=True, random_state=1), s),
           "Boosting": lambda s: baumpipeline(GradientBoostingClassifier(max_depth=2, random_state=0), s)}
print(daten[NEU].isna().sum().tolist(), round(daten.loc[daten["ausfall_naechste"] == 1, "vib_rel"].mean(), 3))

### TODO 8 · Ein rollierendes Merkmal und zwei Fallen
Das Skript nimmt 30 Schichten. Hier baust du einen kurzfristigen Normalwert der Schwingung aus 9 Schichten, mindestens 3, und zwei falsche Fassungen davon.
1. `normal9`: je Maschine der Median der Schwingung der letzten 9 eigenen Schichten ohne die aktuelle, als Spalte zu `daten` passend. `rel9`: die relative Abweichung `vibration_mm_s / normal9 - 1`.
2. `rel9_ohne_gruppe`: dasselbe ohne `groupby`, also über die Zeilen der Datei. `rel9_mitte`: je Maschine mit einem mittigen Fenster über 9 Schichten einschließlich der aktuellen, `rolling(9, min_periods=3, center=True)`, ohne `shift`.
3. `auc9`: ein Wörterbuch mit den Schlüsseln `"richtig"`, `"ohne Gruppe"` und `"mitte"` und der AUC des jeweiligen Merkmals allein im zweiten Jahr, Zeilen mit fehlendem Wert weggelassen.
4. `geaendert9`: ein Wörterbuch mit den Schlüsseln `"richtig"` und `"mitte"`: die Zahl der Zeilen vor dem 1. Juli 2025, deren Wert sich ändert, wenn du das Merkmal nur mit `frueh9` berechnest. Vergleiche mit `np.isclose(..., equal_nan=True)`.

In [ ]:
frueh9 = daten[daten["datum"] < "2025-07-01"]

# TODO 8a
normal9 = None
rel9 = None

# TODO 8b
rel9_ohne_gruppe = None
rel9_mitte = None

# TODO 8c
auc9 = None

# TODO 8d
geaendert9 = None

zeige(auc9, geaendert9, None if rel9 is None else int(rel9.isna().sum()))

In [ ]:
check("TODO 8a Normalwert aus 9 eigenen Schichten", lambda: len(rel9) == 28920 and int(rel9.isna().sum()) == 156 and math.isclose(float(normal9.sum()), 70476.615, abs_tol=1e-3)
      and math.isclose(auc9["richtig"], 0.71277543, abs_tol=1e-6))
check("TODO 8b zwei falsche Fassungen", lambda: math.isclose(auc9["ohne Gruppe"], 0.6706234, abs_tol=1e-6) and math.isclose(auc9["mitte"], 0.71752108, abs_tol=1e-6))
check("TODO 8d Abschneidetest", lambda: geaendert9 == {"richtig": 0, "mitte": 63})

Allein ordnet die Abweichung vom kurzfristigen Normalwert die Schichten des zweiten Jahres mit einer AUC von 0,713. Ohne `groupby` fällt sie auf 0,671: Der Median über die Zeilen der Datei mischt Fräsmaschinen mit Prüfständen. Die mittige Fassung kommt auf 0,718, weil sie die folgenden Schichten kennt, und fällt im Abschneidetest in 63 Zeilen auf. Die richtige Fassung ändert sich nicht.

### TODO 9 · Rohwerte und neue Merkmale zusammen
Im Skript stehen die Rohwerte und die neuen Merkmale getrennt. Was passiert, wenn ein Modell beide bekommt?
1. `p9`: ein Wörterbuch mit den Wahrscheinlichkeiten im zweiten Jahr für die Schlüssel `("logistisch", "Rohwerte")`, `("logistisch", "beide")`, `("Boosting", "Rohwerte")` und `("Boosting", "beide")`. Die Modelle baust du mit `modelle["logistisch"](spalten)` und `modelle["Boosting"](spalten)`, „beide“ heißt `ROH + NEU`. Angepasst wird mit `jahr1[spalten + ["typ"]]` und `y1`.
2. `tab9`: eine Tabelle mit einer Zeile je Schlüssel in derselben Reihenfolge und den Spalten `auc`, `ap`, `alarm_vor` und `alarm_nach`: AUC und mittlere Precision im zweiten Jahr und der Anteil der Schichten an Linie 2 mit $p \ge 0{,}1$ vor und ab dem Sensortausch. Die Masken stehen in `l2_vor` und `l2_nach`.

In [ ]:
# TODO 9a
p9 = None

# TODO 9b
tab9 = None

zeige(tab9)
print(None if tab9 is None else tab9.round(3))

In [ ]:
check("TODO 9a Wahrscheinlichkeiten", lambda: len(p9) == 4 and all(len(v_) == 14580 for v_ in p9.values())
      and math.isclose(float(np.mean(p9[("Boosting", "beide")])), 0.055135, abs_tol=1e-4))
check("TODO 9b Kennzahlen", lambda: np.allclose(tab9[["auc", "ap"]].to_numpy(), [[0.74015195, 0.27281611], [0.75405162, 0.34150826], [0.74003273, 0.26714634], [0.76127389, 0.32961513]], atol=1e-4))
check("TODO 9b Alarme an Linie 2", lambda: np.allclose(tab9[["alarm_vor", "alarm_nach"]].to_numpy(), [[0.08403909, 0.32701422], [0.14397394, 0.19146919], [0.0781759, 0.27203791], [0.11921824, 0.1464455]], atol=1e-7))

Mit beiden Merkmalssätzen liegen AUC und mittlere Precision fast so hoch wie mit den neuen allein im Skript. Die Alarmquote an Linie 2 nach dem Tausch steigt aber wieder: bei der logistischen Regression auf 19,1 %, mit den neuen Merkmalen allein waren es 15,7 %, beim Boosting auf 14,6 % statt 12,8 %. Sobald die Rohwerte im Modell stehen, wirkt der Sprung der neuen Sensoren wieder mit. Wer robuste Modelle will, lässt die Rohwerte weg oder trainiert nach dem Tausch neu.

## 5 · Hyperparameter fair wählen
*Lerneinheit 5 · Skript Abschnitt 5*

Hyperparameter wählst du mit Validierungsdaten, bei Zeitdaten mit `TimeSeriesSplit` im ersten Jahr. `GridSearchCV` probiert jede Kombination und trainiert die beste auf allen Trainingsdaten neu. Ihr bester Wert ist optimistisch, denn er ist der beste von vielen verrauschten. Ob die Gewinnerin in jeder Falte gewinnt und wie viele Einstellungen innerhalb eines Standardfehlers liegen, zeigt, wie sicher die Wahl ist. Das zweite Jahr wird erst am Ende angesehen.

### TODO 10 · Zwei Parameter für die Toleranz suchen
Das Skript sucht für die Härte über Lernrate und Blätter. Hier suchst du für die Toleranz über Tiefe und Blattgröße.
1. `suche_tol`: `GridSearchCV` mit `HistGradientBoostingClassifier(early_stopping=False, random_state=0)`, dem Gitter `gitter_tol`, `cv=TimeSeriesSplit(n_splits=5)` und `scoring="roc_auc"`, angepasst mit dem ersten Jahr für `in_toleranz`.
2. `beste_tol`: die gewählten Parameter, `bester_cv`: deren mittlere AUC in der Kreuzvalidierung, `auc_test_tol`: die AUC des gewählten Modells im zweiten Jahr.
3. `siege_tol`: in wie vielen der fünf Falten die gewählte Einstellung die höchste AUC hatte. Die Werte je Falte stehen in `suche_tol.cv_results_["split0_test_score"]` bis `"split4_test_score"`. `nahe_tol`: wie viele Einstellungen eine mittlere AUC von mindestens `bester_cv` minus einem Standardfehler haben, also minus der Streuung der Gewinnerin über die Falten geteilt durch $\sqrt5$.

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, KFold

gitter_tol = {"max_depth": [2, 3, None], "min_samples_leaf": [10, 20, 50]}

# TODO 10a
suche_tol = None

# TODO 10b
beste_tol = None
bester_cv = None
auc_test_tol = None

# TODO 10c
siege_tol = None
nahe_tol = None

zeige(beste_tol, bester_cv, auc_test_tol, siege_tol, nahe_tol)

In [ ]:
check("TODO 10a Suche mit TimeSeriesSplit", lambda: type(suche_tol.cv).__name__ == "TimeSeriesSplit" and len(suche_tol.cv_results_["params"]) == 9
      and np.allclose(suche_tol.cv_results_["mean_test_score"], [0.81587687, 0.80726119, 0.78217137, 0.84630419, 0.83335889, 0.80673353, 0.8451986, 0.8384454, 0.82564395], atol=1e-6))
check("TODO 10b gewählt im ersten, geprüft im zweiten Jahr", lambda: beste_tol == {"max_depth": 3, "min_samples_leaf": 10} and math.isclose(bester_cv, 0.84630419, abs_tol=1e-6)
      and math.isclose(auc_test_tol, 0.8482141, abs_tol=1e-6))
check("TODO 10c Wie sicher ist die Wahl?", lambda: siege_tol == 1 and nahe_tol == 4)

Gewählt wird Tiefe 3 mit mindestens 10 Chargen je Blatt, mittlere AUC 0,846. Sie gewinnt aber nur in einer von fünf Falten, und vier der neun Einstellungen liegen innerhalb eines Standardfehlers, die unbegrenzte Tiefe mit 10 Chargen kommt auf 0,845. Im zweiten Jahr erreicht die Wahl 0,848. Hättest du nach dem zweiten Jahr gewählt, wäre es die unbegrenzte Tiefe mit 50 Chargen je Blatt und 0,878 gewesen, aber diese Zahl wäre keine ehrliche Prüfung mehr. Die Wahl im ersten Jahr ist unsicher, und das zeigen die Siege und der Standardfehler schon vorher.

## 6 · Was das Modell gelernt hat
*Lerneinheit 6 · Skript Abschnitt 6*

Die **Unreinheits-Wichtigkeit** `feature_importances_` summiert, wie stark die Teilungen eines Merkmals die Unreinheit im Training senken. Sie bevorzugt Merkmale mit vielen möglichen Schwellen. Die **Permutations-Wichtigkeit** vertauscht ein Merkmal in den Prüfdaten und misst, wie stark das Gütemaß fällt. Die **partielle Abhängigkeit** setzt ein Merkmal bei allen Zeilen auf denselben Wert und mittelt die Vorhersagen, die **ICE-Kurven** zeigen dasselbe je Zeile. Alle drei beschreiben das Modell, keine Ursache.

### TODO 11 · Permutations- gegen Unreinheits-Wichtigkeit für die Toleranz
Das Skript misst die Wichtigkeit für die Härte. Hier misst du sie für den Wald `rf_tol`, mit der AUC als Gütemaß.
1. `perm_tol`: `permutation_importance(rf_tol, ...)` auf den Chargen des zweiten Jahres mit `n_repeats=10`, `random_state=5` und `scoring="roc_auc"`.
2. `wichtig_tol`: eine Tabelle mit dem Index `OFEN` und den Spalten `unreinheit` (aus `rf_tol.feature_importances_`) und `permutation` (Mittel aus `perm_tol`), absteigend nach `permutation` sortiert.
3. `perm_train_tol`: dieselbe Rechnung auf den Chargen des ersten Jahres. `oel_verhaeltnis`: die Permutations-Wichtigkeit der Öltemperatur im ersten Jahr geteilt durch die im zweiten.

In [ ]:
from sklearn.inspection import permutation_importance

# TODO 11a
perm_tol = None

# TODO 11b
wichtig_tol = None

# TODO 11c
perm_train_tol = None
oel_verhaeltnis = None

zeige(wichtig_tol, oel_verhaeltnis)
print(None if wichtig_tol is None else wichtig_tol.round(3))

In [ ]:
check("TODO 11a Permutation auf den Prüfdaten", lambda: perm_tol.importances.shape == (6, 10)
      and np.allclose(perm_tol.importances_mean, [0.011, 0.087, 0.0188, 0.18, 0.0258, 0.1856], atol=0.02))
check("TODO 11b Tabelle", lambda: set(wichtig_tol.index[:3]) == {"transfer_s", "kohlenstoff_pct", "haltezeit_min"} and wichtig_tol["permutation"].is_monotonic_decreasing
      and np.allclose(wichtig_tol.loc[OFEN, "unreinheit"], [0.07174541, 0.1570176, 0.06768138, 0.24342348, 0.09009119, 0.37004093], atol=1e-6))
check("TODO 11c Training gegen Prüfung", lambda: np.allclose(perm_train_tol.importances_mean, [0.0238, 0.0983, 0.0218, 0.2233, 0.0525, 0.2117], atol=0.02) and 1.4 <= oel_verhaeltnis <= 2.8)

Nach beiden Maßen führen Transferzeit, Kohlenstoff und Haltezeit. Die Öltemperatur bekommt nach Unreinheit 0,090, mehr als die Hälfte der Haltezeit mit 0,157, nach Permutation auf den Prüfdaten aber nur etwa 0,025, weniger als ein Drittel der Haltezeit. Auf den Trainingsdaten wirkt sie etwa doppelt so wichtig: Dort hat sich der Wald auch Rauschen gemerkt. Die Ofentemperatur kostet beim Vertauschen nur etwa 0,01 AUC.

### TODO 12 · Partielle Abhängigkeit der Öltemperatur
Das Skript zeigt die Transferzeit. Hier untersuchst du die Öltemperatur mit dem Wald `rf` für die Härte.
1. `pd_oel_hand`: für jeden Wert aus `raster_oel` die mittlere Vorhersage von `rf` über die Chargen des zweiten Jahres, wenn bei allen die Öltemperatur auf diesen Wert gesetzt wird, als Array.
2. `pd_oel`: `partial_dependence(rf, h2[OFEN], ["oeltemp_c"], kind="both", method="brute", grid_resolution=10)`. `ice_spanne`: für jede Charge die Differenz ihrer ICE-Kurve zwischen dem letzten und dem ersten Gitterwert, als Array. `anteil_fallend`: der Anteil der Chargen mit negativer Spanne.
3. Zeichne die partielle Abhängigkeit von Hand und die ICE-Kurven der ersten 50 Chargen in eine Grafik.

In [ ]:
from sklearn.inspection import partial_dependence

raster_oel = np.arange(40, 71, 5.0)

# TODO 12a
pd_oel_hand = None

# TODO 12b
pd_oel = None
ice_spanne = None
anteil_fallend = None

# TODO 12c
fig, ax = plt.subplots(figsize=(7, 4))

zeige(None if pd_oel_hand is None else np.round(pd_oel_hand, 2), None if ice_spanne is None else round(float(np.mean(ice_spanne)), 2), anteil_fallend)
plt.show()

In [ ]:
check("TODO 12a partielle Abhängigkeit von Hand", lambda: np.allclose(pd_oel_hand, [56.09103615, 56.0169725, 55.90594035, 55.68231234, 55.34242574, 55.15666655, 55.09087722], atol=5e-3))
check("TODO 12b ICE-Kurven", lambda: pd_oel["individual"][0].shape == (1406, 10) and math.isclose(float(np.mean(ice_spanne)), -0.93945214, abs_tol=5e-3)
      and anteil_fallend > 0.99)

Von 40 auf 70 °C Öltemperatur sinkt die vorhergesagte Härte im Mittel von 56,09 auf 55,09 HRC, um 1 HRC. Fast alle ICE-Kurven fallen, im Mittel um 0,94 HRC über das Gitter. Das ist ein flacher, gleichmäßiger Zusammenhang ohne Knick, wie ihn auch eine Gerade beschreiben könnte. Ob kälteres Öl die Härte erhöht, sagt erst ein Versuch.

## 7 · Wann gewinnen Bäume, wann nicht?
*Lerneinheit 7 · Skript Abschnitt 7*

Bäume finden Schwellen und Wechselwirkungen selbst, extrapolieren aber nicht: Jenseits der Trainingswerte bleibt ihre Vorhersage konstant. Eine faire Entscheidung vergleicht Modelle auf denselben Daten mit denselben Kennzahlen und nimmt das einfachste, das die Messlatte erfüllt.

### TODO 13 · Extrapolation und ein Vergleichsbericht
Das Skript zeigt die Extrapolation für die Transferzeit. Hier lernen zwei Modelle nur Chargen mit höchstens 90 Minuten Haltezeit und sagen Chargen mit mehr als 100 Minuten voraus.
1. `rf13`: `RandomForestRegressor(n_estimators=200, max_features=3, min_samples_leaf=5, random_state=1)` und `lin13`: `LinearRegression()`, beide angepasst mit `innen13` für die Härte. `extra13`: ein Wörterbuch mit den Schlüsseln `"beobachtet"`, `"Wald"` und `"linear"`: die mittlere gemessene Härte der Chargen in `aussen13` und die beiden mittleren Vorhersagen dafür.
2. `bericht`: eine Tabelle mit dem Index `["Mehrheit", "logistisch", "Baum Tiefe 4", "Random Forest", "Boosting"]` und den Spalten `auc` und `brier` für `in_toleranz` im zweiten Jahr. Die Modelle: `DummyClassifier(strategy="prior")`, `make_pipeline(StandardScaler(), LogisticRegression(solver="newton-cholesky"))`, `DecisionTreeClassifier(max_depth=4, random_state=0)`, der fertige Wald `rf_tol` und `HistGradientBoostingClassifier(early_stopping=False, random_state=0)`. Alle außer `rf_tol` passt du mit dem ersten Jahr an.
3. `einfachstes`: der Name des ersten Modells in dieser Reihenfolge, das im zweiten Jahr eine AUC von mindestens 0,85 erreicht.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import brier_score_loss

innen13 = h1[h1["haltezeit_min"] <= 90]
aussen13 = h2[h2["haltezeit_min"] > 100]

# TODO 13a
rf13 = None
lin13 = None
extra13 = None

# TODO 13b
bericht = None

# TODO 13c
einfachstes = None

zeige(extra13, einfachstes)
print(None if bericht is None else bericht.round(4))

In [ ]:
check("TODO 13a Extrapolation", lambda: len(aussen13) == 76 and math.isclose(extra13["beobachtet"], 57.25263158, abs_tol=1e-6)
      and math.isclose(extra13["Wald"], 56.41853282, abs_tol=1e-3) and math.isclose(extra13["linear"], 58.49368215, abs_tol=1e-6))
check("TODO 13b Vergleichsbericht", lambda: list(bericht.index) == ["Mehrheit", "logistisch", "Baum Tiefe 4", "Random Forest", "Boosting"]
      and np.allclose(bericht["auc"], [0.5, 0.73218915, 0.72862825, 0.8690909, 0.87779505], atol=1e-6)
      and np.allclose(bericht["brier"], [0.20003017, 0.17395936, 0.15589952, 0.12296581, 0.12067025], atol=1e-6))
check("TODO 13c einfachstes Modell über der Messlatte", lambda: einfachstes == "Random Forest")

Die Chargen mit mehr als 100 Minuten Haltezeit haben im Mittel 57,25 HRC. Der Wald sagt 56,42 voraus, so viel wie bei 90 Minuten, die Gerade 58,49, weil sie ihre Steigung fortsetzt. Hier liegt die Gerade weiter daneben, im Skript bei der Transferzeit war es umgekehrt: Außerhalb der Daten weiß keines der Modelle Bescheid. Im Bericht erfüllen erst Random Forest und Boosting die Messlatte von 0,85. Die logistische Regression und der Baum der Tiefe 4 bleiben unter 0,74, die Mehrheitsklasse bei 0,5.

### Bonus B1 · Ein Polynom als lineare Antwort
Die logistische Regression bleibt bei 0,73, weil die Toleranz auf zwei Seiten verfehlt wird. Quadrate und Produkte der Merkmale geben ihr Krümmung.
1. `poly_tol`: `make_pipeline(StandardScaler(), PolynomialFeatures(degree=2), LogisticRegression(solver="newton-cholesky"))`, angepasst mit dem ersten Jahr für `in_toleranz`.
2. `n_koef`: die Zahl der Koeffizienten der logistischen Regression in der Pipeline. `auc_poly_tol`: die AUC im zweiten Jahr.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Bonus B1
poly_tol = None
n_koef = None
auc_poly_tol = None

zeige(n_koef, auc_poly_tol)

In [ ]:
check("B1 Polynom zweiten Grades", lambda: n_koef == 28 and math.isclose(auc_poly_tol, 0.87025804, abs_tol=1e-6))

Mit 28 Koeffizienten erreicht das Polynom eine AUC von 0,871, so viel wie der Random Forest. Die Quadrate von Transferzeit und Kohlenstoff bilden beide Seiten der Toleranz ab. Wer die Form kennt, kommt auch mit einem linearen Modell weit.

### Bonus B2 · Kalibrierung des Boosting-Modells
1. `p_b2` und `p_lr_b2`: die Wahrscheinlichkeiten für `in_toleranz` im zweiten Jahr von `HistGradientBoostingClassifier(early_stopping=False, random_state=0)` und von der logistischen Regression `make_pipeline(StandardScaler(), LogisticRegression(solver="newton-cholesky"))`, beide mit dem ersten Jahr angepasst.
2. `kal_b2`: eine Tabelle mit den Spalten `vorhergesagt` und `beobachtet` aus `calibration_curve` mit fünf gleich großen Klassen für das Boosting. `brier_b2`: ein Array mit den Brier-Scores von Boosting und logistischer Regression.

In [ ]:
from sklearn.calibration import calibration_curve

# Bonus B2
p_b2 = None
p_lr_b2 = None
kal_b2 = None
brier_b2 = None

zeige(kal_b2, brier_b2)

In [ ]:
check("B2 Kalibrierung des Boostings", lambda: np.allclose(kal_b2.to_numpy().T, [[0.13654834, 0.6834859, 0.93053172, 0.98136844, 0.99493737],
      [0.23049645, 0.6227758, 0.8683274, 0.93238434, 0.96797153]], atol=1e-7) and np.allclose(brier_b2, [0.12067025, 0.17395936], atol=1e-7))

Das Boosting ist zu sicher: In der obersten Klasse sagt es 99,5 % voraus, beobachtet sind 96,8 %, in der zweithöchsten 98,1 % gegen 93,2 %, in der untersten 13,7 % gegen 23,0 %. Trotzdem ist sein Brier-Score mit 0,121 viel besser als der der logistischen Regression mit 0,174, weil es die Chargen viel besser ordnet. Wer mit den Wahrscheinlichkeiten rechnet, etwa für Kosten wie in Modul 8, rekalibriert sie vorher.

## Zum Nachdenken

1. In TODO 2 liegen bei viel Kohlenstoff und schnellem Transfer nur 40 % der Chargen in der Toleranz. Soll das Werk für solchen Stahl die Transferzeit verlängern? Was müsstest du dafür noch wissen?
2. Eine Kollegin meldet nach TODO 5 stolz eine Out-of-Bag-AUC, die mit dem Datum als Merkmal gestiegen ist. Was antwortest du?
3. In TODO 8 merkt man der mittigen Fassung in der AUC kaum etwas an. Warum ist sie trotzdem falsch, und woran erkennst du solche Merkmale, bevor ein Modell in Betrieb geht?
4. Nach TODO 10 liegen vier Einstellungen innerhalb eines Standardfehlers. Welche würdest du nehmen und warum?
5. Nach TODO 13 erfüllen Random Forest und Boosting die Messlatte, das Polynom aus B1 ebenfalls. Welches Modell schlägst du der Werksleitung vor?

<details><summary>Hinweise zu den Antworten</summary>

1. Der Baum beschreibt einen Zusammenhang in diesen Daten. Ob eine längere Transferzeit die Chargen weicher macht, ohne neue Fehler zu erzeugen, zeigt nur ein geplanter Versuch (Modul 12). Dazu kommt, dass langsamer Transfer bei anderem Stahl mit zu weichen Chargen einhergeht.
2. Out-of-Bag prüft an Schichten aus denselben Wochen wie das Training. Mit dem Datum kann der Wald sich Wochen mit vielen Ausfällen merken, im nächsten Jahr gibt es diese Tage nicht mehr. Ehrlich ist nur eine Prüfung mit späteren Daten.
3. Im Betrieb sind die folgenden Schichten unbekannt, das Merkmal ließe sich gar nicht berechnen. Der Abschneidetest zeigt es: Für Zeilen vor einem Stichtag muss jedes Merkmal gleich bleiben, wenn nur die Daten bis dahin vorliegen.
4. Nach der Regel eines Standardfehlers die einfachste, also Tiefe 3 mit 20 Chargen je Blatt. Sie ist kaum schlechter, schneller und weniger anfällig für Überanpassung.
5. Das einfachste, das die Messlatte erfüllt und sich erklären lässt. Das Polynom hat 28 Koeffizienten und lässt sich als Formel prüfen, der Wald ist robuster gegen ungewöhnliche Einstellungen. Wichtig sind außerdem Kalibrierung, Überwachung und eine Regel, wann neu trainiert wird.
</details>

## Wie es weitergeht

Du hast Bäume von Hand geteilt und gelesen, Bagging und Boosting selbst nachgebaut, zeitlich gestoppt, rollierende Merkmale mit und ohne Leckage gebaut, Hyperparameter fair gesucht, Wichtigkeiten gemessen und die Grenzen der Bäume gesehen. Bisher hatte jede Aufgabe eine Zielgröße. **Modul 10 „Unüberwachtes Lernen“** fragt, was du tun kannst, wenn es keine gibt: Maschinen nach ähnlichem Verhalten gruppieren und ungewöhnliche Schichten finden.